<a href="https://colab.research.google.com/github/Prajkta11222/Iot_Based_Smart_Health_Monetoring_System/blob/main/Vital_Risk_ML_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#Find the dataset automatically
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive")

# Search for CSV files
csv_files = list(BASE.rglob("*.csv"))

print("CSV files found:")
for i, f in enumerate(csv_files):
    print(i, "→", f)

if not csv_files:
    raise FileNotFoundError("No CSV file found in Google Drive.")

# Select your Patient_data dataset
matches = [
    f for f in csv_files
    if "patient" in f.name.lower()
    or "health" in f.name.lower()
    or "monitor" in f.name.lower()
]

print("\nPossible dataset files:")
for i, f in enumerate(matches):
    print(i, "→", f)

DATA_PATH = matches[0]   # change index only if necessary

df = pd.read_csv(DATA_PATH)

print("\nDataset:", DATA_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

CSV files found:
0 → /content/drive/MyDrive/iot_dataset/mitbih_train.csv
1 → /content/drive/MyDrive/iot_dataset/mitbih_test.csv
2 → /content/drive/MyDrive/iot_dataset/Synthetic_patient-HealthCare-Monitoring_dataset.csv

Possible dataset files:
0 → /content/drive/MyDrive/iot_dataset/Synthetic_patient-HealthCare-Monitoring_dataset.csv

Dataset: /content/drive/MyDrive/iot_dataset/Synthetic_patient-HealthCare-Monitoring_dataset.csv
Shape: (60000, 13)
Columns: ['Patient Number', 'Heart Rate (bpm)', 'SpO2 Level (%)', 'Systolic Blood Pressure (mmHg)', 'Diastolic Blood Pressure (mmHg)', 'Body Temperature (°C)', 'Fall Detection', 'Predicted Disease', 'Data Accuracy (%)', 'Heart Rate Alert', 'SpO2 Level Alert', 'Blood Pressure Alert', 'Temperature Alert']


,Patient Number,Heart Rate (bpm),SpO2 Level (%),Systolic Blood Pressure (mmHg),Diastolic Blood Pressure (mmHg),Body Temperature (°C),Fall Detection,Predicted Disease,Data Accuracy (%),Heart Rate Alert,SpO2 Level Alert,Blood Pressure Alert,Temperature Alert
0,1,98,96,120,86,38.1,No,Diabetes Mellitus,95,NORMAL,NORMAL,NORMAL,ABNORMAL
1,2,105,97,177,104,37.6,No,Heart Disease,91,ABNORMAL,NORMAL,ABNORMAL,ABNORMAL
2,3,90,85,139,57,37.0,No,Asthma,98,NORMAL,ABNORMAL,NORMAL,NORMAL
3,4,102,87,101,77,36.4,No,Asthma,87,ABNORMAL,ABNORMAL,NORMAL,NORMAL
4,5,81,95,172,119,37.7,No,Hypertension,88,NORMAL,NORMAL,ABNORMAL,ABNORMAL


In [4]:
#Inspect before training
print("Columns:")
for c in df.columns:
    print(repr(c))

print("\nMissing values:")
print(df.isna().sum())

print("\nData types:")
print(df.dtypes)

print("\nUnique values:")
for c in df.columns:
    if df[c].nunique() < 20:
        print(f"\n{c}:")
        print(df[c].unique())

Columns:
'Patient Number'
'Heart Rate (bpm)'
'SpO2 Level (%)'
'Systolic Blood Pressure (mmHg)'
'Diastolic Blood Pressure (mmHg)'
'Body Temperature (°C)'
'Fall Detection'
'Predicted Disease'
'Data Accuracy (%)'
'Heart Rate Alert'
'SpO2 Level Alert'
'Blood Pressure Alert'
'Temperature Alert'

Missing values:
Patient Number                     0
Heart Rate (bpm)                   0
SpO2 Level (%)                     0
Systolic Blood Pressure (mmHg)     0
Diastolic Blood Pressure (mmHg)    0
Body Temperature (°C)              0
Fall Detection                     0
Predicted Disease                  0
Data Accuracy (%)                  0
Heart Rate Alert                   0
SpO2 Level Alert                   0
Blood Pressure Alert               0
Temperature Alert                  0
dtype: int64

Data types:
Patient Number                       int64
Heart Rate (bpm)                     int64
SpO2 Level (%)                       int64
Systolic Blood Pressure (mmHg)       int64
Diastolic Blo

In [7]:
# STAGE 2 — CREATE ML DATASET
import numpy as np

FEATURES = [
    "Heart Rate (bpm)",
    "SpO2 Level (%)",
    "Body Temperature (°C)"
]

ALERT_COLUMNS = [
    "Heart Rate Alert",
    "SpO2 Level Alert",
    "Blood Pressure Alert",
    "Temperature Alert"
]

data = df[FEATURES + ALERT_COLUMNS].copy()

# Create engineering risk label
data["risk_level"] = np.where(
    data[ALERT_COLUMNS].eq("ABNORMAL").any(axis=1),
    "HIGH",
    "LOW"
)

# Keep only model inputs + target
ml_data = data[FEATURES + ["risk_level"]].copy()

print("Dataset shape:", ml_data.shape)
print("\nClass distribution:")
print(ml_data["risk_level"].value_counts())

display(ml_data.head())

Dataset shape: (60000, 4)

Class distribution:
risk_level
HIGH    47172
LOW     12828
Name: count, dtype: int64


,Heart Rate (bpm),SpO2 Level (%),Body Temperature (°C),risk_level
0,98,96,38.1,HIGH
1,105,97,37.6,HIGH
2,90,85,37.0,HIGH
3,102,87,36.4,HIGH
4,81,95,37.7,HIGH


In [8]:
# STAGE 3 — TRAIN & EVALUATE
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

X = ml_data[FEATURES]
y = ml_data["risk_level"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)

print(classification_report(y_test, pred, digits=4))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))

high_index = list(model.classes_).index("HIGH")
print("ROC-AUC:", round(
    roc_auc_score((y_test == "HIGH").astype(int), prob[:, high_index]), 4
))

              precision    recall  f1-score   support

        HIGH     0.9911    0.9635    0.9771      9434
         LOW     0.8784    0.9680    0.9210      2566

    accuracy                         0.9645     12000
   macro avg     0.9347    0.9658    0.9491     12000
weighted avg     0.9670    0.9645    0.9651     12000

Confusion Matrix:
 [[9090  344]
 [  82 2484]]
ROC-AUC: 0.9887


In [9]:
# STAGE 4 — EXPORT FOR FASTAPI
import joblib, json, os

OUT = "/content/drive/MyDrive/Smart_Health_Monitoring/models"
os.makedirs(OUT, exist_ok=True)

joblib.dump(model, f"{OUT}/vital_risk_model.pkl")

metadata = {
    "model_version": "1.0",
    "features": FEATURES,
    "classes": model.classes_.tolist(),
    "target": "risk_level",
    "label_definition": "HIGH if any source vital alert is ABNORMAL; otherwise LOW",
    "dataset_size": len(ml_data),
    "model": "RandomForestClassifier",
    "random_state": 42
}

with open(f"{OUT}/vital_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved successfully:")
print(f"{OUT}/vital_risk_model.pkl")
print(f"{OUT}/vital_metadata.json")

Saved successfully:
/content/drive/MyDrive/Smart_Health_Monitoring/models/vital_risk_model.pkl
/content/drive/MyDrive/Smart_Health_Monitoring/models/vital_metadata.json


In [10]:
from google.colab import files

files.download("/content/drive/MyDrive/Smart_Health_Monitoring/models/vital_risk_model.pkl")
files.download("/content/drive/MyDrive/Smart_Health_Monitoring/models/vital_metadata.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>